In [1]:
import sys
import json
from tqdm import tqdm
import os

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.qa_pipeline import QAPipeline
from src.qa_pipeline.query_parser import QueryLLMParser
from src.qa_pipeline.knowledge_comparator import KnowledgeComparator
from src.qa_pipeline.knowledge_retriever import KnowledgeRetriever
from src.qa_pipeline.answer_generator import QALLMGenerator

from src.llm_agent import AgentConnector
from src.knowledge_graph_model import KnowledgeGraphModel
from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection

from src.embedding_functions import ChromaConnection, VectorDBConnectionConfig, EmbeddingsDatabaseConnectionConfig

EVAL_DATADIR = '../../data/qa_eval'
VERSION = 'v1'

In [2]:
agent = AgentConnector.open()
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", db_name="testdb"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [3]:
qa_pipeline = QAPipeline(kg_model, agent)

In [4]:
len(gen_answers)

NameError: name 'gen_answers' is not defined

In [7]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in qa_files[8:]:
    print(qa_file)
    with open(f"{EVAL_DATADIR}/{qa_file}", 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())

    gen_answers = []
    process = tqdm(data)
    for qa_pair in process:
        gen_answer = qa_pipeline.answer(qa_pair['question'])
        gen_answers.append({"generated_answer": gen_answer})
        process.set_postfix({'target': qa_pair['answer'], 'generated': gen_answer})
    
    with open(f"./logs/{VERSION}/{qa_file}", 'w', encoding='utf-8') as fd:
        fd.write(json.dumps(gen_answers, indent=1, ensure_ascii=False))

which_people_about_device_synonims.json


  0%|          | 0/116 [00:00<?, ?it/s]

  8%|▊         | 9/116 [18:02<3:34:24, 120.22s/it, target=Abraham, generated=None]   


KeyboardInterrupt: 

In [14]:
from src.utils.evaluation_metrics import ReaderMetrics

In [15]:
METRICS = ReaderMetrics(base_dir=BASEDIR, bs_model_path="google/electra-base-discriminator")

Loading Meteor...
Loading ExactMatch
Loading BertScore


In [17]:
import numpy as np

In [21]:
#gen_answers = list(map(lambda item: item['generated_answer'], gen_answers)) 
trgt_answers = list(map(lambda item: item['answer'], data))[:len(gen_answers)]

b1_scores = METRICS.bleu1(gen_answers, trgt_answers)
b2_scores  = METRICS.bleu2(gen_answers, trgt_answers)
rl_scores = METRICS.rougel(gen_answers, trgt_answers)
m_scores = METRICS.meteor(gen_answers, trgt_answers)
em_scores = METRICS.exact_match(gen_answers, trgt_answers)
bs_scores = METRICS.bertscore(gen_answers, trgt_answers)

scores = {
    'BLEU1': str(round(np.mean(b1_scores),5)),
    'BLEU2': str(round(np.mean(b2_scores),5)),
    'METEOR': str(round(np.mean(m_scores),5)),
    'RougeL': str(round(np.mean(rl_scores),5)),
    'ExactMatch': str(round(np.mean(em_scores),5)),
    'BertScore': {k: str(round(float(v.mean()),5)) for k, v in bs_scores.items() if k != 'hash'}
}

In [22]:
scores

{'BLEU1': '0.63207',
 'BLEU2': '0.53766',
 'METEOR': '0.56857',
 'RougeL': '0.63344',
 'ExactMatch': '0.63021',
 'BertScore': {'precision': '0.77543', 'recall': '0.78506', 'f1': '0.77947'}}